# Topic Modeling of Recent Academic Publications using BERTopic  
## Discovering Hot Research Topics from 2020 to 2025

In [ ]:
%%capture
!pip install bertopic
!pip install umap-learn
!pip install hdbscan
!pip install sentence-transformers
!pip install plotly

In [ ]:
!pip install -qq numpy==1.26.4 gensim
get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Dataset

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/MachineLearning/processed.csv")
print(df.shape)
df.head()

(50812, 14)


,title,abstract,year,timestamp,cleaned_title,cleaned_abstract,title_tokens,title_lemmatized,title_stemmed,abstract_tokens,abstract_lemmatized,abstract_stemmed,combined_lemmatized,combined_stemmed
0,Security Threats and Research Challenges of Io...,Internet of things (IoT) is the epitome of sus...,2020,2025-04-09 00:01:06,security threats and research challenges of io...,internet of things iot is the epitome of susta...,"['security', 'threats', 'challenges', 'iota']","['security', 'threat', 'challenge', 'iota']","['secur', 'threat', 'challeng', 'iota']","['internet', 'things', 'iot', 'epitome', 'sust...","['internet', 'thing', 'iot', 'epitome', 'susta...","['internet', 'thing', 'iot', 'epitom', 'sustai...",security threat challenge iota internet thing ...,secur threat challeng iota internet thing iot ...
1,Routing Approach for P2P Systems Over MANET Ne...,Thanks to the great progress in mobile and wir...,2020,2025-04-09 00:01:06,routing approach for pp systems over manet net...,thanks to the great progress in mobile and wir...,"['routing', 'systems', 'manet', 'network']","['rout', 'system', 'manet', 'network']","['rout', 'system', 'manet', 'network']","['thanks', 'great', 'progress', 'mobile', 'wir...","['thanks', 'great', 'progress', 'mobile', 'wir...","['thank', 'great', 'progress', 'mobil', 'wirel...",rout system manet network thanks great progres...,rout system manet network thank great progress...
2,Predicting Individual Substance Abuse Vulnerab...,Substance abuse is the unrestrained and detrim...,2020,2025-04-09 00:01:06,predicting individual substance abuse vulnerab...,substance abuse is the unrestrained and detrim...,"['predicting', 'individual', 'substance', 'abu...","['predict', 'individual', 'substance', 'abuse'...","['predict', 'individu', 'substanc', 'abus', 'v...","['substance', 'abuse', 'unrestrained', 'detrim...","['substance', 'abuse', 'unrestrained', 'detrim...","['substanc', 'abus', 'unrestrain', 'detriment'...",predict individual substance abuse vulnerabili...,predict individu substanc abus vulner machin l...
3,Explainability Matters: Backdoor Attacks on Me...,Deep neural networks have been shown to be vul...,2020,2025-04-09 00:01:06,explainability matters backdoor attacks on med...,deep neural networks have been shown to be vul...,"['explainability', 'matters', 'backdoor', 'att...","['explainability', 'matter', 'backdoor', 'atta...","['explain', 'matter', 'backdoor', 'attack', 'm...","['deep', 'neural', 'networks', 'vulnerable', '...","['deep', 'neural', 'network', 'vulnerable', 'b...","['deep', 'neural', 'network', 'vulner', 'backd...",explainability matter backdoor attack medical ...,explain matter backdoor attack medic imag deep...
4,Encoding sinusoidal functions in hybrid automa...,Hybrid systems can express a plethora of physi...,2020,2025-04-09 00:01:06,encoding sinusoidal functions in hybrid automa...,hybrid systems can express a plethora of physi...,"['encoding', 'sinusoidal', 'functions', 'hybri...","['encode', 'sinusoidal', 'function', 'hybrid',...","['encod', 'sinusoid', 'function', 'hybrid', 'a...","['hybrid', 'systems', 'express', 'plethora', '...","['hybrid', 'system', 'express', 'plethora', 'p...","['hybrid', 'system', 'express', 'plethora', 'p...",encode sinusoidal function hybrid automaton fo...,encod sinusoid function hybrid automata formal...


In [ ]:
df = df.dropna(subset=['combined_lemmatized'])
documents = df['combined_lemmatized'].tolist()
years = df["year"].astype(str).tolist()
print(f"Total documents: {len(documents)}")
print("First document sample:")
print(documents[0][:300], "...")

Total documents: 50812
First document sample:
security threat challenge iota internet thing iot epitome sustainable development facilitate development smart system industrialization quality life iot architecture essential baseline understand widespread adoption security issue crucial technical infrastructure since iot comprises heterogeneous de ...


##2. Modeling

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

### 2.1. Vectorizer Model

In [ ]:
vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),
    stop_words="english",
    min_df=5
)

### 2.2. Embedding Model

In [ ]:
embedding_model = SentenceTransformer("paraphrase-mpnet-base-v2")

### 2.3. BERTopic Model

In [ ]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    min_topic_size=50,
    calculate_probabilities=False,
    verbose=True
)

#### 2.3.1. Training

In [ ]:
import time

start = time.time()
topics, probs = topic_model.fit_transform(documents)
end = time.time()
duration = end - start
print(f"Training completed in {duration:.2f} seconds.")

2025-06-19 13:12:39,757 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/1588 [00:00<?, ?it/s]

2025-06-19 13:14:40,467 - BERTopic - Embedding - Completed ✓
2025-06-19 13:14:40,468 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-06-19 13:15:17,603 - BERTopic - Dimensionality - Completed ✓
2025-06-19 13:15:17,605 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-06-19 13:15:21,642 - BERTopic - Cluster - Completed ✓
2025-06-19 13:15:21,654 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-06-19 13:15:31,995 - BERTopic - Representation - Completed ✓


Training completed in 175.72 seconds.


In [ ]:
save_path = "/content/drive/MyDrive/bertopic_model"
topic_model.save(save_path)
print("Model saved to Google Drive.")

2025-06-19 13:15:42,364 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


Model saved to Google Drive.


In [ ]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,22742,-1_model_learn_language_image,"[model, learn, language, image, performance, n...",[promptdistill querybased selective token rete...
1,0,1886,0_robot_control_robotic_grasp,"[robot, control, robotic, grasp, manipulation,...",[stochastic assignment deploy marsupial robot ...
2,1,1024,1_channel_communication_wireless_mimo,"[channel, communication, wireless, mimo, trans...",[quantum sense joint beam training uavmounted ...
3,2,806,2_graph_vertex_algorithm_bound,"[graph, vertex, algorithm, bound, tree, time, ...",[tight bound connectivity parameterized cutwid...
4,3,698,3_speech_speaker_asr_speech recognition,"[speech, speaker, asr, speech recognition, rec...",[automate speech recognition system conversati...
...,...,...,...,...,...
121,120,57,120_covid_tweet_twitter_pandemic,"[covid, tweet, twitter, pandemic, social mediu...",[explore hybrid deep learn framework automatic...
122,121,56,121_unlearn_machine unlearn_forget_machine,"[unlearn, machine unlearn, forget, machine, re...",[towards machine unlearn benchmark forget pers...
123,122,54,122_hallucination_llm_factual_language model,"[hallucination, llm, factual, language model, ...",[delucionqa detect hallucination domainspecifi...
124,123,52,123_anomaly_anomaly detection_video_detection,"[anomaly, anomaly detection, video, detection,...",[weaklysupervised anomaly detection surveillan...


## 3. Visualizations

In [ ]:
fig = topic_model.visualize_documents(documents)

for i in range(len(fig.data)):
    fig.data[i].text = None

fig.show()

In [ ]:
topic_model.visualize_topics()

In [ ]:
topics_over_time = topic_model.topics_over_time(documents, years)
topic_model.visualize_topics_over_time(topics_over_time)

### Topics over Time - Growing Topics from 2020 to 2025

In [ ]:
topics_over_time["Year"] = pd.to_datetime(topics_over_time["Timestamp"]).dt.year.astype(str)
pivot = topics_over_time.pivot_table(
    index="Year", columns="Topic", values="Frequency", aggfunc="sum"
)

first_year = pivot.index.min()
last_year = pivot.index.max()
growth = pivot.loc[last_year] - pivot.loc[first_year]

top_growing_topics = growth.sort_values(ascending=False).head(10).index.tolist()

In [ ]:
for topic_id in top_growing_topics:
    words = topic_model.get_topic(topic_id)
    keywords = [word for word, _ in words]
    print(f"Topic {topic_id}: {', '.join(keywords)}")

Topic 5: diffusion, diffusion model, image, edit, generation, style, texttoimage, model, video, image generation
Topic 21: reason, llm, language model, prompt, language, large language, cot, reason task, answer, model llm
Topic 19: visual, mllms, vlms, visionlanguage, vqa, multimodal, reason, token, answer, question
Topic 54: gaussian, splatting, gaussian splatting, dg, render, scene, gaussians, reconstruction, view, splatting dg
Topic 18: code, bug, program, software, code generation, test, llm, developer, language, generation
Topic 35: retrieval, rag, query, document, search, retrievalaugmented, llm, retrieve, retriever, answer
Topic 36: video, temporal, caption, moment, video understand, video caption, long video, understand, retrieval, videotext
Topic 91: motion, human motion, motion generation, human, generation, motion synthesis, animation, diffusion, motion sequence, video
Topic 64: vision transformer, transformer, vits, vit, vision, token, attention, transformer vits, image, pa

#### Growing Topics from 2020 to 2025


| Topic ID | Description                               |
|----------|-------------------------------------------------|
| 5        | Diffusion-Based Image & Video Generation         |
| 21       | Reasoning & Prompting in LLMs                     |
| 19       | Multimodal Question Answering (VQA)               |
| 54       | 3D Rendering via Gaussian Splatting                |
| 18       | Code Generation & Software Debugging               |
| 35       | Retrieval-Augmented Generation (RAG)               |
| 36       | Video Understanding & Captioning                    |
| 91       | Human Motion Synthesis & Animation                   |
| 64       | Vision Transformers (ViTs)                            |
| 113      | Parameter-Efficient Fine-Tuning (LoRA, PEFT)          |

In [ ]:
topic_model.visualize_hierarchy()

In [ ]:
topic_model.visualize_heatmap()

## 4. Performance Metrics

### 4.1. Coherence Scores

In [ ]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary

tokenized_docs = [doc.split() for doc in documents]
dictionary = Dictionary(tokenized_docs)
corpus = [dictionary.doc2bow(text) for text in tokenized_docs]

topics = []
for topic_num in topic_model.get_topics().keys():
    topic = topic_model.get_topic(topic_num)
    if topic is not None:
        words = [word for word, _ in topic]
        topics.append(words)

coherence_cv = CoherenceModel(topics=topics, texts=tokenized_docs, dictionary=dictionary, coherence='c_v').get_coherence()
coherence_umass = CoherenceModel(topics=topics, texts=tokenized_docs, dictionary=dictionary, coherence='u_mass').get_coherence()
coherence_npmi = CoherenceModel(topics=topics, texts=tokenized_docs, dictionary=dictionary, coherence='c_npmi').get_coherence()
coherence_uci = CoherenceModel(topics=topics, texts=tokenized_docs, dictionary=dictionary, coherence='c_uci').get_coherence()

print(f"C_v Coherence:     {coherence_cv:.4f}")
print(f"U_Mass Coherence: {coherence_umass:.4f}")
print(f"NPMI Coherence:   {coherence_npmi:.4f}")
print(f"UCI Coherence:    {coherence_uci:.4f}")

C_v Coherence:     0.7437
U_Mass Coherence: -2.3801
NPMI Coherence:   0.1999
UCI Coherence:    1.8053


### 4.2. PUW

In [ ]:
all_words = [word for topic in topics for word in topic]

unique_words = set(all_words)
puw = len(unique_words) / len(all_words)

print(f"Proportion of Unique Words (PUW): {puw:.4f}")

Proportion of Unique Words (PUW): 0.7405


### 4.3. Avg. Jaccard Similarity

In [ ]:
from itertools import combinations

def jaccard_similarity(set1, set2):
    return len(set1 & set2) / len(set1 | set2)

jaccard_scores = []
for t1, t2 in combinations(topics, 2):
    jaccard_scores.append(jaccard_similarity(set(t1), set(t2)))

avg_jaccard = sum(jaccard_scores) / len(jaccard_scores)
print(f"Average Jaccard Similarity between topics: {avg_jaccard:.4f}")

Average Jaccard Similarity between topics: 0.0055
